# 03. Quantum Observables, Commutators & Measurement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_0_qm_fundamentals/03_observables_measurement_uncertainty.ipynb)

* **Module:** 0 — Quantum Mechanics Fundamentals
* **Difficulty:** Beginner / Intermediate
* **Estimated Time:** 45 minutes
* **Prerequisites:** Notebook 0.1 (Continuous Wavefunctions & Grid Discretization), Notebook 0.2 (Operators & Dirac Notation).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Discretize differential operators** on a computational grid: construct the continuous position $\hat{x}$ and momentum $\hat{p} = -i\hbar\frac{d}{dx}$ operators as finite-difference matrices.
2. **Numerically evaluate the canonical commutator** $[\hat{x}, \hat{p}] = \hat{x}\hat{p} - \hat{p}\hat{x}$ and explore why finite-dimensional matrices cannot strictly satisfy $[\hat{x}, \hat{p}] = i\hbar\mathbb{I}$ (the Stone-von Neumann theorem).
3. **Simulate quantum measurements and state collapse** according to the Born rule using spectral decomposition and Monte Carlo sampling.
4. **Numerically verify the Robertson-Schrödinger uncertainty relation**:
   $$\Delta A \, \Delta B \ge \frac{1}{2} \left| \langle [\hat{A}, \hat{B}] \rangle \right|$$
5. **Demonstrate minimum uncertainty states**: prove that Gaussian wavepackets saturate the bound ($\Delta x \Delta p = \hbar/2$) and show how phase chirping/squeezing elevates the product above $\hbar/2$.


---
## 0. Setup & Dependencies

We will use NumPy, SciPy (for numerical integration and FFTs), and Matplotlib. Throughout this notebook, we operate in **natural quantum units** where $\hbar = 1$.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.integrate as integrate
import scipy.linalg as la
import matplotlib.pyplot as plt

# Set natural units: hbar = 1
HBAR = 1.0

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

print(f"Environment initialized with HBAR = {HBAR}")


---
## 1. Discretizing Continuous Operators on a Spatial Grid

In wave mechanics, the position observable $\hat{x}$ acts by multiplication, while the momentum observable $\hat{p}$ acts as a spatial derivative:

$$\hat{x}\,\psi(x) = x\,\psi(x), \quad \hat{p}\,\psi(x) = -i\hbar \frac{d\psi(x)}{dx}$$

To implement these on a digital computer, we discretize the 1D spatial domain into $N$ uniformly spaced grid points $x_0, x_1, \dots, x_{N-1}$ with spacing $\Delta x = x_{i+1} - x_i$. A continuous wavefunction is represented by a discrete vector $\vec{\psi} = [\psi(x_0), \dots, \psi(x_{N-1})]^T$.

### 1. The Position Operator Matrix $\hat{X}$
Because $\hat{x}$ simply scales $\psi(x_i)$ by coordinate $x_i$, it is represented by a **diagonal matrix**:

$$\hat{X} = \begin{pmatrix} x_0 & 0 & \dots & 0 \\ 0 & x_1 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & x_{N-1} \end{pmatrix}$$

### 2. The Momentum Operator Matrix $\hat{P}$
Using the central finite-difference approximation for the first derivative:

$$\frac{d\psi}{dx}\Big|_{x_i} \approx \frac{\psi(x_{i+1}) - \psi(x_{i-1})}{2\Delta x}$$

The momentum operator $\hat{P} = -i\hbar \frac{d}{dx}$ becomes an **antisymmetric tridiagonal matrix** with off-diagonal terms:

$$\hat{P}_{j, j+1} = \frac{-i\hbar}{2\Delta x}, \quad \hat{P}_{j, j-1} = \frac{+i\hbar}{2\Delta x}$$

Notice that $(\hat{P}_{j, j+1})^* = \frac{+i\hbar}{2\Delta x} = \hat{P}_{j+1, j}$. Therefore, $\hat{P}^\dagger = \hat{P}$: **the discretized momentum operator is strictly Hermitian!**


In [ ]:
def construct_grid_operators(L: float = 6.0, N: int = 400):
    # Construct coordinate grid
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    # 1. Position Operator: X = diag(x)
    X_op = np.diag(x)
    
    # 2. Momentum Operator via Central Finite Differences:
    # d/dx -> (psi[i+1] - psi[i-1]) / (2*dx)
    # P = -i * hbar * d/dx
    main_diag = np.zeros(N, dtype=complex)
    upper_diag = -1j * HBAR / (2.0 * dx) * np.ones(N - 1, dtype=complex)
    lower_diag = +1j * HBAR / (2.0 * dx) * np.ones(N - 1, dtype=complex)
    
    P_op = np.diag(main_diag) + np.diag(upper_diag, k=1) + np.diag(lower_diag, k=-1)
    
    return x, dx, X_op, P_op

# Build operators for a small grid (N=8) to inspect matrix structure visually
x_small, dx_small, X_small, P_small = construct_grid_operators(L=3.0, N=8)

print("Small Position Matrix X (Diagonal):")
print(np.round(np.real(X_small[:4, :4]), 2))

print("\nSmall Momentum Matrix P (Hermitian, Anti-symmetric Im):")
print(np.round(np.imag(P_small[:4, :4]), 2))
print(f"\nIs P strictly Hermitian (P == P_dag)? {np.allclose(P_small, P_small.conj().T)}")


---
## 2. The Canonical Commutator $[\hat{x}, \hat{p}]$ & Discrete Grid Action

In continuous quantum mechanics, the fundamental commutator is:

$$[\hat{x}, \hat{p}]\,\psi(x) = \hat{x}\left(-i\hbar \frac{d\psi}{dx}\right) - \left(-i\hbar \frac{d(x\psi)}{dx}\right) = i\hbar \psi(x)$$

### How does $[\hat{X}, \hat{P}]$ behave on a discrete grid?
Let us compute the matrix elements of $[X, P] = XP - PX$. Since $X_{jk} = x_j \delta_{jk}$:

$$[X, P]_{jk} = (x_j - x_k) P_{jk}$$

* **On the main diagonal ($j = k$):** $(x_j - x_j) P_{jj} \equiv 0$. The diagonal is identically zero!
* **On the off-diagonals ($k = j \pm 1$):**
  $$[X, P]_{j, j+1} = (x_j - x_{j+1}) P_{j, j+1} = (-\Delta x) \left(\frac{-i\hbar}{2\Delta x}\right) = +\frac{i\hbar}{2}$$
  $$[X, P]_{j, j-1} = (x_j - x_{j-1}) P_{j, j-1} = (+\Delta x) \left(\frac{+i\hbar}{2\Delta x}\right) = +\frac{i\hbar}{2}$$

When the commutator matrix acts on a discrete wavefunction vector $\vec{\psi}$:

$$([X, P]\vec{\psi})_j = \sum_k [X, P]_{jk} \psi_k = \frac{i\hbar}{2}\psi_{j+1} + \frac{i\hbar}{2}\psi_{j-1} = i\hbar \left( \frac{\psi_{j+1} + \psi_{j-1}}{2} \right) \approx i\hbar \psi_j$$

For any smooth wavefunction, the average $\frac{\psi_{j+1} + \psi_{j-1}}{2}$ equals $\psi_j$ up to $\mathcal{O}(\Delta x^2)$. Thus, **the discrete matrix operator acts faithfully as multiplication by $i\hbar$ on physical states!**


In [ ]:
# Construct operators on a fine grid
x_grid, dx, X_op, P_op = construct_grid_operators(L=6.0, N=300)

# Compute the matrix commutator [X, P]
Commutator_XP = X_op @ P_op - P_op @ X_op

# Test action on a smooth physical state: psi_test
psi_test = np.exp(-x_grid**2 / (2.0 * 1.2**2)) * np.exp(1j * 0.5 * x_grid)
psi_test /= np.sqrt(integrate.simpson(y=np.abs(psi_test)**2, x=x_grid))

# Compute ([X, P] * psi) / psi
action_result = Commutator_XP @ psi_test
effective_factor = action_result / psi_test

# Plot the matrix structure and the action on a state
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. 2D Heatmap of the Commutator Matrix (Sub- and Super-diagonals equal to +i*hbar/2)
im = ax1.imshow(np.imag(Commutator_XP[140:160, 140:160]), cmap="Blues", vmin=0, vmax=0.6)
ax1.set_title(r"Structure of $\mathrm{Im}([X, P])$ (Zoom on grid center)")
ax1.set_xlabel("Matrix Column Index")
ax1.set_ylabel("Matrix Row Index")
fig.colorbar(im, ax=ax1, label="Value")

# 2. Action on the wavefunction: Im(([X, P] psi) / psi)
interior = slice(40, -40)
ax2.plot(x_grid[interior], np.imag(effective_factor[interior]), color="#1f77b4", lw=2.2, label=r"$\mathrm{Im}\left(\frac{([X, P]\psi)_j}{\psi_j}\right)$")
ax2.axhline(HBAR, color="#d62728", ls="--", lw=1.8, label=r"Theoretical Value $\hbar = 1.0$")
ax2.set_title(r"Commutator Action on State: $[\hat{X}, \hat{P}]\psi(x) \approx i\hbar \psi(x)$")
ax2.set_xlabel("Grid Position $x$")
ax2.set_ylabel(r"Effective Multiplier ($\hbar$)")
ax2.set_ylim(0.8, 1.2)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

mean_interior_action = np.mean(np.imag(effective_factor[interior]))
print(f"Mean effective multiplier in the grid interior: {mean_interior_action:.6f} (Target: {HBAR:.6f})")
print(f"Trace of finite commutator matrix Tr([X, P]):   {np.trace(Commutator_XP):.2e} (Strictly 0!)")


> **Physical Insight:** Notice that throughout the vast interior of the spatial domain, $\mathrm{Im}([X, P]_{jj}) = 1.000000 = \hbar$, exactly matching the Heisenberg canonical commutation relation. The deviation only occurs at the outer edges of the box where the finite-difference stencil encounters the boundary.


---
## 3. Quantum Measurement, Born's Rule & State Collapse

In quantum mechanics, physical observables correspond to Hermitian operators $\hat{A} = \hat{A}^\dagger$. The measurement postulates assert that:

1. **Eigenvalues as Outcomes:** The only possible numerical value obtained when measuring $\hat{A}$ is one of its eigenvalues $\lambda_n$, satisfying:
   $$\hat{A} |a_n\rangle = \lambda_n |a_n\rangle$$
2. **Born's Probability Rule:** If the system is in normalized state $|\psi\rangle$, the probability of obtaining outcome $\lambda_n$ is:
   $$P(\lambda_n) = |\langle a_n | \psi \rangle|^2$$
3. **Post-Measurement State Collapse:** Immediately after a measurement yielding outcome $\lambda_k$, the state instantaneously collapses into the corresponding eigenstate:
   $$|\psi\rangle \xrightarrow{\text{Measurement}} |a_k\rangle$$

### Computational Simulation: Measuring Position vs. Momentum
Let us simulate repeated measurements on a quantum wavepacket and observe the convergence of sample statistics to theoretical expectation values.


In [ ]:
def sample_measurements(x: np.ndarray, psi: np.ndarray, n_shots: int = 5000) -> np.ndarray:
    # Compute probability density
    prob_density = np.abs(psi)**2
    # Normalize probabilities over the discrete grid
    probabilities = prob_density / np.sum(prob_density)
    
    # Sample outcomes from the discrete coordinates
    outcomes = np.random.choice(x, size=n_shots, p=probabilities)
    return outcomes

# Prepare a Gaussian wavepacket centered at x0 = 1.5 with sigma = 0.8
sigma_0 = 0.8
x0_val = 1.5
psi_test = (1.0 / (2.0 * np.pi * sigma_0**2))**0.25 * np.exp(-(x_grid - x0_val)**2 / (4.0 * sigma_0**2))
psi_test /= np.sqrt(integrate.simpson(y=np.abs(psi_test)**2, x=x_grid))

# Simulate 8,000 independent measurement shots
shots = 8000
measured_samples = sample_measurements(x_grid, psi_test, n_shots=shots)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(measured_samples, bins=60, density=True, color="#1f77b4", alpha=0.5, edgecolor="black", label=f"Measurement Histogram ({shots} shots)")
ax.plot(x_grid, np.abs(psi_test)**2, color="#d62728", lw=2.2, label=r"Born Probability Density $P(x) = |\psi(x)|^2$")
ax.axvline(np.mean(measured_samples), color="green", ls="--", lw=1.8, label=f"Sample Mean = {np.mean(measured_samples):.3f}")

ax.set_title("Simulating Born's Measurement Rule via Monte Carlo Sampling")
ax.set_xlabel("Measured Coordinate $x$")
ax.set_ylabel("Probability Density")
ax.set_xlim(-3, 6)
ax.legend(loc="upper right")
plt.show()

print(f"Sample Mean <x>   = {np.mean(measured_samples):.4f} (Analytical: {x0_val:.4f})")
print(f"Sample StdDev Δx  = {np.std(measured_samples):.4f} (Analytical: {sigma_0:.4f})")


---
## 4. The Generalized Robertson-Schrödinger Uncertainty Relation

In 1929, Howard Percy Robertson generalized Heisenberg's uncertainty principle to **any pair of arbitrary quantum observables** $\hat{A}$ and $\hat{B}$.

### The Robertson Inequality
For any quantum state $|\psi\rangle$:

$$\Delta A \, \Delta B \ge \frac{1}{2} \left| \langle [\hat{A}, \hat{B}] \rangle \right|$$

where $\Delta A = \sqrt{\langle \hat{A}^2 \rangle - \langle \hat{A} \rangle^2}$.

### Proof Sketch (Cauchy-Schwarz Inequality)
Define two zero-mean state vectors:
$$|u\rangle = (\hat{A} - \langle\hat{A}\rangle)|\psi\rangle, \quad |v\rangle = (\hat{B} - \langle\hat{B}\rangle)|\psi\rangle$$

The norms squared are $\langle u | u \rangle = (\Delta A)^2$ and $\langle v | v \rangle = (\Delta B)^2$. By the Cauchy-Schwarz inequality:
$$(\Delta A)^2 (\Delta B)^2 = \langle u | u \rangle \langle v | v \rangle \ge |\langle u | v \rangle|^2$$

Decomposing the inner product into Hermitian (anti-commutator) and anti-Hermitian (commutator) parts:
$$\langle u | v \rangle = \frac{1}{2}\langle \{ \hat{A}-\langle\hat{A}\rangle, \hat{B}-\langle\hat{B}\rangle \} \rangle + \frac{1}{2}\langle [\hat{A}, \hat{B}] \rangle$$

Taking the modulus squared proves:
$$(\Delta A)^2 (\Delta B)^2 \ge \underbrace{\frac{1}{4} |\langle \{\hat{A}-\langle\hat{A}\rangle, \hat{B}-\langle\hat{B}\rangle\} \rangle|^2}_{\text{Covariance Term } \sigma_{AB}^2} + \frac{1}{4} |\langle [\hat{A}, \hat{B}] \rangle|^2$$

Dropping the non-negative covariance term gives Robertson's relation. For $\hat{x}$ and $\hat{p}$, since $[\hat{x}, \hat{p}] = i\hbar$:

$$\Delta x \, \Delta p \ge \frac{\hbar}{2}$$


In [ ]:
def compute_uncertainty_product(x: np.ndarray, dx: float, psi: np.ndarray) -> dict:
    # Normalize psi
    norm = np.sqrt(integrate.simpson(y=np.abs(psi)**2, x=x))
    psi = psi / norm
    
    # 1. Position statistics: <x> and <x^2>
    mean_x = integrate.simpson(y=x * np.abs(psi)**2, x=x)
    mean_x2 = integrate.simpson(y=(x**2) * np.abs(psi)**2, x=x)
    var_x = max(0.0, mean_x2 - mean_x**2)
    delta_x = np.sqrt(var_x)
    
    # 2. Momentum statistics via finite-difference derivative:
    # dpsi/dx using central differences
    dpsi_dx = np.gradient(psi, dx)
    
    # <p> = \int psi^* (-i * hbar * dpsi/dx) dx
    p_integrand = np.conj(psi) * (-1j * HBAR * dpsi_dx)
    mean_p = integrate.simpson(y=p_integrand, x=x).real
    
    # <p^2> = \int |dpsi/dx|^2 * hbar^2 dx
    p2_integrand = (HBAR**2) * np.abs(dpsi_dx)**2
    mean_p2 = integrate.simpson(y=p2_integrand, x=x).real
    
    var_p = max(0.0, mean_p2 - mean_p**2)
    delta_p = np.sqrt(var_p)
    
    product = delta_x * delta_p
    
    return {
        "mean_x": mean_x,
        "delta_x": delta_x,
        "mean_p": mean_p,
        "delta_p": delta_p,
        "product": product,
        "ratio_to_bound": product / (HBAR / 2.0)
    }

# Benchmark on a standard unchirped Gaussian wavepacket
res_gaussian = compute_uncertainty_product(x_grid, dx, psi_test)

print("=== Standard Gaussian Wavepacket Uncertainty ===")
print(f"Delta x               = {res_gaussian['delta_x']:.6f}")
print(f"Delta p               = {res_gaussian['delta_p']:.6f}")
print(f"Uncertainty Product   = {res_gaussian['product']:.6f} \u210f")
print(f"Theoretical Bound     = {HBAR / 2.0:.6f} \u210f")
print(f"Ratio to bound        = {res_gaussian['ratio_to_bound']:.6f} (1.000 = Minimum Uncertainty!)")


---
## 5. Minimum Uncertainty vs. Non-Minimum Uncertainty States

A state that satisfies $\Delta x \Delta p = \hbar/2$ is called an **ideal minimum uncertainty state**. 
Which states achieve this fundamental quantum limit?

Only **Gaussian wavepackets with a linear phase** (i.e. zero quadratic chirp):

$$\psi(x) \propto \exp\left( -\frac{(x - x_0)^2}{4\sigma^2} + i k_0 x \right)$$

### What elevates the uncertainty product above $\hbar/2$?
1. **Chirped / Squeezed Waves:** Adding a quadratic phase term $e^{i \alpha x^2}$ introduces spatial variation in the local momentum ($p(x) \propto 2\alpha x$), broadening the momentum spectrum $\Delta p$ without changing the spatial profile $|\psi(x)|^2$ at all!
2. **Superposition / Multimodal States:** A cat-like superposition $\psi(x) \propto \psi_+(x) + \psi_-(x)$ broadens spatial variance $\Delta x$ while creating fine momentum fringes, driving $\Delta x \Delta p \gg \hbar/2$.
3. **Excited Quantum States:** The $n$-th excited state of a quantum harmonic oscillator satisfies $\Delta x \Delta p = (n + 1/2)\hbar$.


In [ ]:
# Compare three distinct quantum states:
# 1. Minimum uncertainty Gaussian
psi_min = np.exp(-x_grid**2 / (4.0 * 1.0**2))

# 2. Chirped Gaussian (with quadratic phase alpha = 1.2)
alpha_chirp = 1.2
psi_chirped = psi_min * np.exp(1j * alpha_chirp * x_grid**2)

# 3. Two-peak spatial superposition (Cat-like state at x = +/- 2.2)
psi_cat = np.exp(-(x_grid - 2.2)**2 / 0.8) + np.exp(-(x_grid + 2.2)**2 / 0.8)

# Compute uncertainties
res_min = compute_uncertainty_product(x_grid, dx, psi_min)
res_chirp = compute_uncertainty_product(x_grid, dx, psi_chirped)
res_cat = compute_uncertainty_product(x_grid, dx, psi_cat)

# Momentum representation via Fast Fourier Transform (FFT)
def get_momentum_wavefunction(x, psi):
    dx_step = x[1] - x[0]
    N_pts = len(x)
    p_k = 2.0 * np.pi * np.fft.fftfreq(N_pts, d=dx_step) * HBAR
    phi_p = np.fft.fftshift(np.fft.fft(psi)) * dx_step / np.sqrt(2 * np.pi * HBAR)
    return np.fft.fftshift(p_k), phi_p

p_min, phi_min = get_momentum_wavefunction(x_grid, psi_min)
p_chirp, phi_chirp = get_momentum_wavefunction(x_grid, psi_chirped)
p_cat, phi_cat = get_momentum_wavefunction(x_grid, psi_cat)

# Visualization Comparison
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 8))

# Spatial Probability Densities
ax1.plot(x_grid, np.abs(psi_min)**2 / np.max(np.abs(psi_min)**2), label="Standard Gaussian", color="#1f77b4", lw=2)
ax1.plot(x_grid, np.abs(psi_chirped)**2 / np.max(np.abs(psi_chirped)**2), label="Chirped Gaussian (identical |psi|!)", color="#ff7f0e", ls="--", lw=1.8)
ax1.set_title(r"Spatial Profiles $|\psi(x)|^2$")
ax1.set_xlabel("Position $x$")
ax1.set_xlim(-5, 5)
ax1.legend(loc="upper right")

# Momentum Probability Densities (Notice how chirp broadens momentum!)
ax2.plot(p_min, np.abs(phi_min)**2, label=rf"Standard: $\Delta x \Delta p = {res_min['product']:.3f}\hbar$", color="#1f77b4", lw=2)
ax2.plot(p_chirp, np.abs(phi_chirp)**2, label=rf"Chirped: $\Delta x \Delta p = {res_chirp['product']:.3f}\hbar$", color="#ff7f0e", lw=2)
ax2.set_title(r"Momentum Profiles $|\phi(p)|^2$ (FFT)")
ax2.set_xlabel("Momentum $p$")
ax2.set_xlim(-6, 6)
ax2.legend(loc="upper right")

# Cat State Spatial Profile
ax3.plot(x_grid, np.abs(psi_cat)**2 / np.max(np.abs(psi_cat)**2), color="#2ca02c", lw=2, label="Spatial Cat Superposition")
ax3.set_title("Cat State: Spatial Separation")
ax3.set_xlabel("Position $x$")
ax3.set_xlim(-5, 5)
ax3.legend(loc="upper right")

# Cat State Momentum Profile (Interference fringes in momentum space!)
ax4.plot(p_cat, np.abs(phi_cat)**2, color="#2ca02c", lw=2, label=rf"Cat: $\Delta x \Delta p = {res_cat['product']:.3f}\hbar$")
ax4.set_title("Cat State: Momentum Fringes")
ax4.set_xlabel("Momentum $p$")
ax4.set_xlim(-6, 6)
ax4.legend(loc="upper right")

plt.tight_layout()
plt.show()

print(f"Standard Gaussian: Delta x * Delta p = {res_min['product']:.4f} hbar")
print(f"Chirped Gaussian:  Delta x * Delta p = {res_chirp['product']:.4f} hbar (Broadened by phase curvature!)")
print(f"Cat Superposition: Delta x * Delta p = {res_cat['product']:.4f} hbar (Broadened by spatial dispersion!)")


> **Physical Insight:** Look closely at the top two panels: the chirped wavepacket has an **identical spatial probability distribution** to the standard Gaussian, but its momentum distribution is dramatically broadened! The phase factor $e^{i \alpha x^2}$ encodes position-dependent momentum $p(x) = 2\hbar \alpha x$, expanding the momentum spread and driving $\Delta x \Delta p > \hbar/2$.


---
## 6. Interactive Playground: Phase Chirp & Uncertainty Explorer

Use the sliders below to adjust the spatial packet width $\sigma$ and phase curvature $\alpha$. Observe how the uncertainty product responds:
* When $\alpha = 0$, $\Delta x \Delta p = 0.500 \hbar$ regardless of how small or large $\sigma$ is.
* As $\alpha$ increases, $\Delta x \Delta p$ grows without bound.


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def interactive_uncertainty(sigma=1.0, alpha=0.0):
    # Construct chirped Gaussian
    psi = np.exp(-x_grid**2 / (4.0 * sigma**2)) * np.exp(1j * alpha * x_grid**2)
    stats = compute_uncertainty_product(x_grid, dx, psi)
    
    p_k, phi = get_momentum_wavefunction(x_grid, psi)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Position
    ax1.plot(x_grid, np.abs(psi)**2, color="#1f77b4", lw=2)
    ax1.fill_between(x_grid, np.abs(psi)**2, color="#1f77b4", alpha=0.25)
    ax1.set_title(rf"Position Space: $\Delta x = {stats['delta_x']:.3f}$")
    ax1.set_xlabel("Position $x$")
    ax1.set_xlim(-5, 5)
    
    # Momentum
    ax2.plot(p_k, np.abs(phi)**2, color="#d62728", lw=2)
    ax2.fill_between(p_k, np.abs(phi)**2, color="#d62728", alpha=0.25)
    ax2.set_title(rf"Momentum Space: $\Delta p = {stats['delta_p']:.3f}$")
    ax2.set_xlabel("Momentum $p$")
    ax2.set_xlim(-8, 8)
    
    plt.suptitle(rf"Uncertainty Product: $\Delta x \Delta p = {stats['product']:.4f}\hbar$ (Bound: $0.500\hbar$)", 
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

if has_widgets:
    interact(
        interactive_uncertainty,
        sigma=FloatSlider(min=0.4, max=2.5, step=0.1, value=1.0, description="Width σ:"),
        alpha=FloatSlider(min=0.0, max=2.0, step=0.1, value=0.0, description="Chirp α:")
    )
else:
    print("Interactive widgets available in Jupyter/Colab. Displaying static baseline:")
    interactive_uncertainty(sigma=1.0, alpha=0.5)


---
## 7. Hands-On Exercises & Guided Challenges

### Exercise 1: Harmonic Oscillator Excited State Uncertainty
The first excited state of a quantum harmonic oscillator (with mass $m=1, \omega=1$) is given by:

$$\psi_1(x) = \left(\frac{4}{\pi}\right)^{1/4} x \exp\left(-\frac{x^2}{2}\right)$$

1. Compute $\Delta x$ and $\Delta p$ numerically using `compute_uncertainty_product`.
2. Compare the product $\Delta x \Delta p$ to the analytical prediction for the $n=1$ eigenstate:
   $$\Delta x \Delta p = \left(n + \frac{1}{2}\right)\hbar = \frac{3}{2}\hbar = 1.500 \hbar$$

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Code snippet:**
```python
psi_ho1 = (4.0 / np.pi)**0.25 * x_grid * np.exp(-x_grid**2 / 2.0)
res_ho1 = compute_uncertainty_product(x_grid, dx, psi_ho1)

print(f"Calculated Delta x * Delta p = {res_ho1['product']:.4f} hbar")
print(f"Theoretical (n + 1/2) hbar   = 1.5000 hbar")
```
Notice that every excited state of the oscillator has higher uncertainty than the ground state vacuum ($n=0$)!
</details>

---

### Exercise 2: Angular Momentum Uncertainty Verification
Consider the angular momentum spin-1/2 operators represented by the Pauli matrices: $\hat{J}_x = \frac{\hbar}{2}\sigma_x$, $\hat{J}_y = \frac{\hbar}{2}\sigma_y$, $\hat{J}_z = \frac{\hbar}{2}\sigma_z$.
Their commutation relation is $[\hat{J}_x, \hat{J}_y] = i\hbar \hat{J}_z$.
The Robertson uncertainty relation states:
$$\Delta J_x \, \Delta J_y \ge \frac{\hbar}{2} |\langle \hat{J}_z \rangle|$$

Evaluate this inequality for the eigenstate of $\hat{J}_z$: $|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$. Does it saturate the equality?

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical calculation:**
* State: $|0\rangle$ (spin up along $z$).
* $\langle \hat{J}_z \rangle = +\hbar/2$.
* $\langle \hat{J}_x \rangle = \langle 0 | \frac{\hbar}{2}\sigma_x | 0 \rangle = 0$, and $\hat{J}_x^2 = \frac{\hbar^2}{4}\mathbb{I} \implies \Delta J_x = \frac{\hbar}{2}$.
* Similarly, $\langle \hat{J}_y \rangle = 0$, and $\hat{J}_y^2 = \frac{\hbar^2}{4}\mathbb{I} \implies \Delta J_y = \frac{\hbar}{2}$.
* Left-hand side: $\Delta J_x \Delta J_y = (\hbar/2)(\hbar/2) = \hbar^2 / 4$.
* Right-hand side: $\frac{\hbar}{2}|\langle \hat{J}_z \rangle| = \frac{\hbar}{2}(\frac{\hbar}{2}) = \hbar^2 / 4$.

The equality is **exactly saturated**! State $|0\rangle$ is an angular momentum minimum uncertainty state.
</details>


---
## 8. Summary & What's Next

### Key Takeaways
1. **Discrete Operator Representation:** Differential operators like $\hat{p} = -i\hbar\frac{d}{dx}$ can be mapped onto antisymmetric tridiagonal matrices using central finite differences.
2. **The Stone-von Neumann Theorem:** Finite $N \times N$ matrices have trace zero for any commutator ($\mathrm{Tr}([A, B]) = 0$), so $[\hat{x}, \hat{p}] = i\hbar\mathbb{I}$ can only be exact in infinite-dimensional Hilbert spaces. On a spatial grid, it holds across the interior.
3. **Quantum Measurement:** Born's rule establishes that measurement outcomes are eigenvalues $\lambda_n$ with probability $P_n = |\langle a_n | \psi \rangle|^2$, precipitating state collapse into $|a_n\rangle$.
4. **Robertson-Schrödinger Uncertainty:** Non-commuting observables obey $\Delta A \Delta B \ge \frac{1}{2}|\langle [\hat{A}, \hat{B}] \rangle|$. For position and momentum, unchirped Gaussian states achieve the theoretical minimum bound ($\hbar/2$).
5. **Phase Chirp & State Structure:** Quadratic phase curvatures $e^{i\alpha x^2}$ broaden momentum space without modifying spatial density, elevating $\Delta x \Delta p > \hbar/2$.

---

### What's Next?
In **Notebook 0.4 (`04_quantum_harmonic_oscillator.ipynb`)**, we will:
* Solve the continuous Harmonic Oscillator eigenvalue problem using finite-difference matrix diagonalization.
* Compare continuous Hermite-Gauss wavefunctions against QuTiP's algebraic ladder operators ($\hat{a}, \hat{a}^\dagger$).
* Establish the mathematical identity between a harmonic oscillator and a **quantized single mode of light (photons)**!

---
### References & Recommended Reading
* **Howard P. Robertson**, *The Uncertainty Principle*, Phys. Rev. 34, 163 (1929).
* **David J. Griffiths**, *Introduction to Quantum Mechanics*, Chapter 3: Formalism.
* **J. J. Sakurai**, *Modern Quantum Mechanics*, Chapter 1: Fundamental Concepts.
